In [0]:
available_coupons_path = (
    "gs://market-intelligence-raw/ebay/"
    "item_details__available_coupons/"
)

available_coupons_df = (
    spark.read
    .option("multiLine", "false")
    .json(available_coupons_path + "*.jsonl.gz")
)

print("Total rows:", available_coupons_df.count())
print("Total columns:", len(available_coupons_df.columns))

available_coupons_df.printSchema()

In [0]:
# =====================================================
# Coupon cardinality per listing
# =====================================================

from pyspark.sql import functions as F

coupon_counts = (
    available_coupons_df
    .groupBy("_dlt_parent_id")
    .agg(
        F.count("*").alias("coupon_count")
    )
)

display(
    coupon_counts
    .agg(
        F.count("*").alias("parent_count"),
        F.avg("coupon_count").alias("avg_coupons"),
        F.expr("percentile_approx(coupon_count, 0.25)").alias("p25"),
        F.expr("percentile_approx(coupon_count, 0.50)").alias("median"),
        F.expr("percentile_approx(coupon_count, 0.75)").alias("p75"),
        F.min("coupon_count").alias("min_coupons"),
        F.max("coupon_count").alias("max_coupons")
    )
)

In [0]:
# =====================================================
# Validate coupon → Item Details relationship
# =====================================================

item_details_path = (
    "gs://market-intelligence-raw/ebay/item_details/"
)

item_details_df = (
    spark.read
    .option("multiLine", "false")
    .json(item_details_path + "*.jsonl.gz")
)

coupon_parent_ids = (
    available_coupons_df
    .select(F.col("_dlt_parent_id").alias("parent_dlt_id"))
    .distinct()
)

item_parent_ids = (
    item_details_df
    .select(F.col("_dlt_id").alias("parent_dlt_id"))
    .distinct()
)

matched_parents = (
    coupon_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="inner")
    .count()
)

unmatched_parents = (
    coupon_parent_ids
    .join(item_parent_ids, on="parent_dlt_id", how="left_anti")
    .count()
)

print("Coupon parent IDs:", coupon_parent_ids.count())
print("Matched to current Item Details:", matched_parents)
print("Not found in current Item Details:", unmatched_parents)

In [0]:
# =====================================================
# Coupon discount distribution
# =====================================================

display(
    available_coupons_df
    .groupBy("discount_type")
    .agg(
        F.count("*").alias("record_count"),
        F.countDistinct("_dlt_parent_id").alias("listing_count")
    )
    .orderBy(F.desc("record_count"))
)

In [0]:
# =====================================================
# Coupon discount completeness
# =====================================================

available_coupons_df.agg(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("discount_amount__value").isNull(), 1)
         .otherwise(0)
    ).alias("discount_value_null"),

    F.sum(
        F.when(F.col("discount_amount__currency").isNull(), 1)
         .otherwise(0)
    ).alias("discount_currency_null"),

    F.sum(
        F.when(F.col("constraint__expiration_date").isNull(), 1)
         .otherwise(0)
    ).alias("expiration_date_null")
).show()

In [0]:
# =====================================================
# Validate coupon expiration dates
# =====================================================

coupon_dates = (
    available_coupons_df
    .select(
        F.to_timestamp("constraint__expiration_date").alias("expiration_date")
    )
)

print(
    "Invalid expiration dates:",
    coupon_dates.filter(
        F.col("expiration_date").isNull()
    ).count()
)

In [0]:
# =====================================================
# Blank / whitespace-only string check
# =====================================================

string_columns = [
    field.name
    for field in available_coupons_df.schema.fields
    if field.dataType.simpleString() == "string"
]

blank_profile = available_coupons_df.agg(*[
    F.sum(
        F.when(
            F.col(c).isNotNull() &
            (F.trim(F.col(c)) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in string_columns
])

blank_counts = blank_profile.first().asDict()

blank_data = [
    (column, int(blank_counts[column]))
    for column in string_columns
    if blank_counts[column] > 0
]

if blank_data:
    display(
        spark.createDataFrame(
            blank_data,
            ["column_name", "blank_count"]
        )
    )
else:
    print("No empty or whitespace-only string values found.")

# Available Coupons Profile

The `available_coupons` dataset contains **1,382 coupon records** across **10 columns**:

- `_dlt_id`
- `_dlt_list_idx`
- `_dlt_parent_id`
- `constraint__expiration_date`
- `discount_amount__currency`
- `discount_amount__value`
- `discount_type`
- `message`
- `redemption_code`
- `terms_web_url`

The dataset represents coupon information associated with Item Details listings, with `_dlt_parent_id` linking each coupon record to its parent listing.

## Coupon Cardinality per Listing

| Metric | Result |
|---|---:|
| Listings with coupons | 1,382 |
| Average coupons per listing | 1 |
| P25 | 1 |
| Median | 1 |
| P75 | 1 |
| Minimum coupons | 1 |
| Maximum coupons | 1 |

Every listing represented in this dataset has exactly **one coupon record**. There are no listings with multiple coupon records in the current snapshot.

This makes the relationship effectively **one-to-one for listings that have a coupon**:

```text
Item Details
    │
    │ 1
    │
    └────────── 0..1 Available Coupon
```
### Coupon → Item Details Relationship

| Validation | Result |
|---|---:|
| Coupon parent IDs | 1,382 |
| Matched to current Item Details | 1,382 |
| Not found in current Item Details | 0 |

All 1,382 coupon parent IDs successfully matched an existing Item Details record.

There are therefore 0 orphaned coupon records in the current snapshot.

### Discount Type Distribution

| Discount Type | Record Count | Listing Count |
|---|---:|---:|
| `ITEM_PRICE` | 1,382 | 1,382 |

All available coupons currently use the `ITEM_PRICE` discount type.

There is therefore no observed variation in `discount_type` in this snapshot. The field should still be retained because the source schema allows the possibility of additional discount types in future data.


### Coupon Data Completeness

| Field | Null Count |
|---|---:|
| `discount_amount__value` | 0 |
| `discount_amount__currency` | 0 |
| `constraint__expiration_date` | 0 |

All 1,382 coupon records have:
- a populated discount amount value
- a populated discount currency
- a populated expiration date
This indicates that the core coupon attributes required for interpreting the discount are fully populated in the current snapshot.

### Expiration Date Validation
The expiration date was converted to a timestamp and validated.

| Validation | Result |
|---|---:|
| Invalid / unparseable expiration dates | 0 |

All coupon expiration dates successfully parsed into timestamps.

Therefore, the Silver layer can safely represent:

`constraint__expiration_date` → timestamp

No special date correction is required based on the current profiling results.


### Blank / Whitespace Validation

No empty or whitespace-only string values were found across the string columns of the available_coupons dataset.
This means there is no evidence of hidden blank-string values that would need to be converted to NULL during the initial Silver transformation.

### Key Findings
1. Small and well-structured child dataset
   The dataset contains only 1,382 records, and every coupon belongs to a valid Item Details listing. There are no orphaned parent references.
2. One coupon per listing in the current snapshot
   Every coupon-bearing listing has exactly one coupon record. The observed cardinality is therefore 1:1 between the parent listing and its coupon record, while the overall Item Details → Coupon relationship remains optional (0..1).
3. Discount information is complete
   `discount_amount__value`, `discount_amount__currency`, and `constraint__expiration_date` contain no NULL values. These fields can therefore be treated as reliable inputs for Silver-layer transformation.
4. Only `ITEM_PRICE` discounts are currently observed
   All 1,382 coupons use `ITEM_PRICE`. We should preserve the raw categorical value rather than hard-coding assumptions about the full possible enumeration.
5. No immediate data-quality issues detected
   There are no orphan records, missing core discount fields, invalid expiration dates, or blank/whitespace-only string values in the current snapshot.


Silver Layer Implications

The coupon dataset can remain as a separate child table rather than being flattened into the main Item Details table.

Recommended transformations:

```text
_dlt_parent_id
    → retain as parent listing relationship

discount_amount__value
    → cast to numeric/decimal

discount_amount__currency
    → retain as currency code

discount_type
    → categorical string

constraint__expiration_date
    → timestamp

message
    → string

redemption_code
    → string

terms_web_url
    → string
```